## FRAP analyses - interpretation

___

In [ ]:
from pathlib import Path
from microlive.pipelines.pipeline_FRAP import *
from microlive.imports import *
from microlive import microscopy as mi
current_dir = Path().resolve()
import scipy.stats as stats
from scipy.stats import ttest_ind
from itertools import combinations
from statsmodels.stats.multitest import multipletests
one_drive_dir = mi.Utilities.get_one_drive_dir()

# Import plotting functions from module
from frap_plotting_module import (
    plot_FRAP_trajectories,
    plot_mean_trajectories_all,
    plot_box_swarm_final_values,
    plot_box_swarm_fit_results,
)

In [ ]:
#results_main_folder = one_drive_dir.joinpath('General - Zhao (NZ) Lab/Microscope/Rhiannon (microscope)/LaG16 Binding Stiochiometry/results_rs')
results_main_folder = one_drive_dir.joinpath('General - Zhao (NZ) Lab/Microscope/Rhiannon (microscope)/LaG16 Binding Stiochiometry/results_la')
cwd = Path.cwd()
results_main_folder = cwd.joinpath('results')
process_single_days = False  # Set to False to combine datasets from multiple days

In [ ]:
# List all immediate subfolders in results_main_folder
subfolders = [folder for folder in results_main_folder.iterdir() if folder.is_dir()]

if process_single_days:
    subfolder_strings = {'sf_d1' :  ["20250709 pRS046", "None"] ,
                        'sf_d2' :  ["20250820 pRS046", "None"] ,
                        'uv_d1': ["20250820 pRS056", "None"],
                        'uv_d2': ["20250828 pRS056", "None"]} 
    list_datasets  = ['sf_d1','sf_d2','uv_d1','uv_d2']
    datasets_to_process = ['sf_d1', 'sf_d2', 'uv_d1', 'uv_d2']
else:
    subfolder_strings = {'sf' :  ["pRS046", "None"] ,
                        'uv': ["pRS056", "None"]} 
    list_datasets  = ['sf','uv']
    datasets_to_process = ['sf', 'uv']


In [ ]:
frap_time = 10 # seconds

In [ ]:
results_folder = results_main_folder.joinpath('test_results_la') 
results_folder.mkdir(exist_ok=True)

# Min–Max Normalization

We apply the following equation to normalize the values in the `mean_roi_frap_normalized` column:

$\text{normalized\_value} = \frac{\text{value} - \text{min\_val}}{\text{max\_val} - \text{min\_val}}$

In [ ]:
apply_quality_check = True
drop_threshold = 0.4
apply_min_max_normalization = True  # Set to False to skip normalization

# Define your dataset types and initialize lists

list_df_paths = []
list_df_FRAP = []
total_number_cells = 0
selected_field = 'mean_roi_frap' # mean_roi_frap_normalized
# Iterate over each dataset type
for dataset_type in list_datasets:
    # Loop through each subfolder (assumes 'subfolders' is defined)
    for subfolder in subfolders:
        # Check if the subfolder name matches one of the strings for this dataset type
        if (subfolder_strings[dataset_type][0] in subfolder.stem) or (subfolder_strings[dataset_type][1] in subfolder.stem):
            # Get CSV files that don't contain the unwanted substrings
            csv_files = [f for f in subfolder.glob("*.csv") 
                         if "no_roi_detected" not in f.name and "df_FRAP_fit" not in f.name]
            if csv_files:
                # Use the first matching CSV file (you may modify this logic if needed)
                selected_csv_path = csv_files[0]
                selected_df = pd.read_csv(selected_csv_path)
                # Keep only the necessary columns
                selected_df_copy = selected_df[['frame', selected_field, 'image_name']].copy()
                
                # Add new columns: dataset_type and subfolder_id
                selected_df_copy['dataset_type'] = dataset_type
                selected_df_copy['subfolder_id'] = subfolder.stem
                
                # Create a unique cell_id for each unique image (cell)
                unique_cells = selected_df_copy['image_name'].unique()
                cell_id_map = {cell: f"{subfolder.stem}_{i}" for i, cell in enumerate(unique_cells, start=1)}
                selected_df_copy['cell_id'] = selected_df_copy['image_name'].map(cell_id_map)
                
                # If quality check is enabled, filter cells based on drop threshold (using unnormalized data)
                if apply_quality_check:
                    filtered_df = pd.DataFrame()
                    for cell in selected_df_copy['image_name'].unique():
                        cell_data = selected_df_copy[selected_df_copy['image_name'] == cell]
                        # Extract data for the first 20 seconds
                        subset_data = cell_data[cell_data['frame'] <= 20]
                        if subset_data.empty:
                            
                            continue
                        initial_intensity = subset_data[selected_field].iloc[0]
                        min_intensity = subset_data[selected_field].min()
                        drop = initial_intensity - min_intensity
                        # Only keep cells with a drop greater than the threshold
                        if drop > drop_threshold:
                            filtered_df = pd.concat([filtered_df, cell_data], ignore_index=True)
                        else:
                            print(f"Cell {cell} in {subfolder.stem} dropped due to insufficient drop ({drop:.2f})")
                    selected_df_copy = filtered_df.copy()
                
                # Optionally apply min–max normalization per cell (after quality check)
                if apply_min_max_normalization:
                    normalized_list = []
                    for cell in selected_df_copy['image_name'].unique():
                        cell_data = selected_df_copy[selected_df_copy['image_name'] == cell].copy()
                        min_val = cell_data[selected_field].min()
                        max_val = cell_data[selected_field].max()
                        if max_val > min_val:
                            cell_data[selected_field] = (
                                cell_data[selected_field] - min_val
                            ) / (max_val - min_val)
                        else:
                            cell_data[selected_field] = 0.0
                        normalized_list.append(cell_data)
                    selected_df_copy = pd.concat(normalized_list, ignore_index=True)
                
                # Append the selected data if not empty
                if not selected_df_copy.empty:
                    list_df_paths.append(selected_csv_path)
                    list_df_FRAP.append(selected_df_copy)
                    total_number_cells += selected_df_copy['cell_id'].nunique()

# Combine all extracted DataFrames into a single dataset
combined_df = pd.concat(list_df_FRAP, ignore_index=True)

# (Optional) Print out the total number of processed rows/cells
print("Combined DataFrame shape:", combined_df.shape)
print("Total number of processed cells:", total_number_cells)

In [ ]:
combined_df

In [ ]:
# Process both datasets
all_fit_results = []

for dataset_type in datasets_to_process:
    print(f"\n{'='*50}")
    print(f"Processing dataset: {dataset_type}")
    print(f"{'='*50}")
    
    df_sel = combined_df[combined_df['dataset_type'] == dataset_type]
    # Get unique combinations of subfolder_id and image_name to avoid mixing data from different experiments
    unique_cells = df_sel[['subfolder_id', 'image_name', 'cell_id']].drop_duplicates()
    print(f"Found {len(unique_cells)} unique cells in '{dataset_type}' dataset")

    # iterate for each unique cell (combination of subfolder_id and image_name)
    for _, row in unique_cells.iterrows():
        subfolder_id = row['subfolder_id']
        image_name = row['image_name']
        cell_id = row['cell_id']
        
        # Filter by both subfolder_id and image_name to get data for this specific cell
        df_image = df_sel[(df_sel['subfolder_id'] == subfolder_id) & (df_sel['image_name'] == image_name)]
        print(f"Processing cell: {cell_id} (subfolder: {subfolder_id}, image: {image_name}) with {len(df_image)} frames")
        
        try:
            # Fit FRAP model
            t_half_single, t_half_double_1st_process, t_half_double_2nd_process, r_squared_single, r_squared_double = fit_model_to_frap(
                time=df_image['frame'], 
                intensity=df_image['mean_roi_frap'],
                frap_time=frap_time, 
                suptitle=None, 
                save_plot=False, 
                plot_name=None
            )
            
            print(f"  Results - Single exp t½: {t_half_single:.2f}, R²: {r_squared_single:.3f}")
            print(f"           Double exp t½1: {t_half_double_1st_process:.2f}, t½2: {t_half_double_2nd_process:.2f}, R²: {r_squared_double:.3f}")
            
        except Exception as e:
            print(f"  Error fitting cell {cell_id}: {e}")
            # Set NaN values if fitting fails
            t_half_single = t_half_double_1st_process = t_half_double_2nd_process = np.nan
            r_squared_single = r_squared_double = np.nan
        
        print("-" * 50)

        # Append results to the list
        all_fit_results.append({
            'dataset_type': dataset_type,
            'subfolder_id': subfolder_id,
            'image_name': image_name,
            'cell_id': cell_id,
            't_half_single': t_half_single,
            'r_squared_single': r_squared_single,
            't_half_double_1st_process': t_half_double_1st_process,
            't_half_double_2nd_process': t_half_double_2nd_process,
            'r_squared_double': r_squared_double
        })

# Create DataFrame from all results
df_all_fit_results = pd.DataFrame(all_fit_results)

# Display summary
print(f"\n{'='*50}")
print("SUMMARY")
print(f"{'='*50}")
print(f"Total cells processed: {len(df_all_fit_results)}")
print("\nCells per dataset:")
print(df_all_fit_results['dataset_type'].value_counts())


In [ ]:
# Group by dataset_type and count unique cell_id values
cell_counts = combined_df.groupby('dataset_type')['cell_id'].nunique()
# Print the total number of processed cells for each dataset type
for dataset_type, count in cell_counts.items():
    print(f"{dataset_type}: {count} cells processed")

In [ ]:
if process_single_days:
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='sf_d1', results_folder=results_folder)
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='sf_d2', results_folder=results_folder)
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='uv_d1', results_folder=results_folder)
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='uv_d2', results_folder=results_folder)
else:
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='sf', results_folder=results_folder)
    plot_FRAP_trajectories(df_list=combined_df, selected_dataset='uv', results_folder=results_folder)
    

In [ ]:
plot_mean_trajectories_all(combined_df, 
                            #['sf', 'uv'],
                            list_datasets,
                            selected_field='mean_roi_frap',
                            apply_quality_check=True,
                            drop_threshold=0.8,
                            apply_min_max_normalization=True,
                            # color_map = ['green' , 'darkgreen', 'black', 'grey'] ,
                            color_map = ['darkgreen', 'black'] ,
                            fig_size=(7,4),
                            use_sem=False, results_folder=results_folder)

In [ ]:
plot_box_swarm_final_values(
    df=combined_df,
    selected_field=selected_field,
    figsize=(4, 4),
    ylabel= "Normalized final recovery \n intensity",
    title="",
    y_min=0,
    #y_max=1.75,
    swarm_color="black",
    tick_size=14,
    order_categories = list_datasets, # ['sf_d1','sf_d2','uv_d1','uv_d2'],
    #order_categories = ['sf','uv'],
    show_stats=True,
    results_folder=results_folder,
)

In [ ]:
# Plot t_half_single comparison
plot_box_swarm_fit_results(
    df=df_all_fit_results,
    selected_field="t_half_single",
    figsize=(4, 4),
    ylabel=r"$t_{1/2}$ (s)",
    title="",
    y_min=0,
    order_categories =  list_datasets, # ['sf_d1','sf_d2','uv_d1','uv_d2'],
    #order_categories = ['sf','uv'],
    y_max=None,  # Let it auto-scale based on your data
    swarm_color="black",
    tick_size=14,
    show_stats=True,
    results_folder=results_folder,
)